# AIML-02 — ChestX-ray14 classifier (Colab T4)

Trains DenseNet-121 on the **official NIH ChestX-ray14 sample** (5,606 images, 14 findings), with a patient-level split so the same patient never appears in train and test.

- Runtime: ~10–20 minutes on a free T4 for 5 epochs.
- Data: Kaggle `nih-chest-xrays/sample` (2.2 GB), fetched with your own token.
- Output: `model.pt` + `metrics.json` (per-label AUROC, mean AUROC, sensitivity/specificity).

> Education only. Not a medical device.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 2. Kaggle token

In Colab, open the key icon (Secrets), add `KAGGLE_API_TOKEN`, and enable notebook access. Do **not** paste the token into a cell.


In [ ]:
import os

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!pip install -q kaggle
!kaggle datasets list --page-size 1 >/dev/null && echo 'kaggle auth OK'

## 3. Clone the submission repo

In [ ]:
!rm -rf /content/nexvia
!git clone --depth 1 https://github.com/OmkarAla/nexvia.git /content/nexvia
!ls /content/nexvia/AIML-02

## 4. Download the NIH sample (5,606 images)

In [ ]:
!mkdir -p /content/nih
!kaggle datasets download -d nih-chest-xrays/sample -p /content/nih --unzip
!ls /content/nih/sample | head
!python -c "import pandas as pd; d=pd.read_csv('/content/nih/sample/sample_labels.csv'); print(len(d), 'rows'); print(d['Finding Labels'].str.split('|').explode().value_counts().head(8))" 

## 5. Train

5 epochs, batch 32, ImageNet-pretrained DenseNet-121. The validation split selects the best epoch; test metrics are reported on untouched patients.


In [ ]:
!cd /content/nexvia && python AIML-02/training/train_classifier.py   --csv /content/nih/sample/sample_labels.csv   --images /content/nih/sample/images   --out /content/artifacts   --epochs 5   --batch-size 32   --learning-rate 1e-4   --num-workers 2   --run-id colab-nih-sample-001

## 6. Inspect the metrics

In [ ]:
import json

metrics = json.load(open("/content/artifacts/metrics.json"))
print("mean AUROC:", round(metrics["mean_auroc"], 4))
print("rows train/val/test:", metrics["n_train"], metrics["n_val"], metrics["n_test"])
print("seconds:", metrics["train_seconds"])
for label, auc in sorted(
    metrics["per_label_auroc"].items(), key=lambda kv: (kv[1] is None, -(kv[1] or 0))
):
    print(f"{label:20s} {auc if auc is None else round(auc, 4)}")

## 7. Download the artifacts

Upload `model.pt` and `metrics.json` into `AIML-02/artifacts/demo/` (or attach them to a GitHub Release if they are large). Keep the run ID with every reported number.


In [ ]:
!cd /content && zip -r imaging-artifacts.zip artifacts
from google.colab import files

files.download("/content/imaging-artifacts.zip")

## Optional: full dataset

The full NIH archive is ~42 GB and needs Kaggle competition-style acceptance. Prefer the sample for the demo; run the full set only if the T4 session has enough disk and time:

```bash
kaggle datasets download -d nih-chest-xrays/data -p /content/nih-full --unzip
```

Then point `--csv` at `Data_Entry_2017.csv` and `--images` at `images/`.
